# 흉부 X-ray 기반 폐렴 분류 모델

Kaggle Chest X-Ray Pneumonia 데이터셋을 활용해 MobileNetV2 기반 전이학습으로
흉부 X-ray 이미지를 정상/폐렴 이진 분류하는 모델입니다.

- Base Model: MobileNetV2 (ImageNet 가중치 고정)
- Head: GlobalAveragePooling + Dropout(0.3) + Dense(1, sigmoid)
- Augmentation: RandomFlip, RandomRotation
- Result: Test Accuracy **82.69%**


## 1. 라이브러리 임포트

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping

## 2. 데이터셋 로드

Kaggle Chest X-Ray Pneumonia 데이터셋을 train/val/test로 분리해 불러옵니다.

In [ ]:
data_dir = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray"
IMG_SIZE = (128, 128)
BATCH_SIZE = 16

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    f"{data_dir}/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    f"{data_dir}/val",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)
test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    f"{data_dir}/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',
    shuffle=False
)

## 3. 전처리: 정규화 및 프리페치

픽셀 값을 [0, 1] 범위로 정규화하고, `AUTOTUNE`으로 파이프라인 성능을 최적화합니다.

In [ ]:
normalization_layer = layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

## 4. 데이터 증강

적은 데이터로도 일반화 성능을 확보하기 위해 간단한 증강 기법을 추가합니다.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
])

## 5. 모델 구성: MobileNetV2 전이학습

가중치를 고정한 MobileNetV2를 베이스로 사용하고, GlobalAveragePooling + Dropout(0.3) + Dense(sigmoid)로 분류 헤드를 구성합니다.

In [ ]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(*IMG_SIZE, 3)
)
base_model.trainable = False  # freeze

inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model = models.Model(inputs, outputs)

## 6. 컴파일

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

## 7. 학습

`EarlyStopping`으로 검증 성능이 정체되면 학습을 조기 종료하고 최적 가중치를 복원합니다.

In [ ]:
early_stop = EarlyStopping(patience=2, restore_best_weights=True)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[early_stop]
)

## 8. 테스트셋 평가

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
"""
Result:
loss: 0.5421 - accuracy: 0.7235 (last training epoch)
Test Accuracy: 0.8269
"""

## Key Learnings

- 계산화학 위주였던 기존 연구 경험에서 벗어나, CNN 구조와 전이학습 개념을 이미지 기반 프로젝트로 직접 체득
- 딥러닝 프로젝트를 처음 다뤄보는 만큼, 무겁고 복잡한 모델보다 빠르게 완결까지 경험할 수 있는 가벼운 구조(MobileNetV2 + 단순 헤드)를 선택
- 낯선 프레임워크(TensorFlow/Keras)와 전이학습 기법을 독학으로 익혀 데이터 준비부터 평가까지 프로젝트를 처음부터 끝까지 완결
